# Zadanie 2.4: skrajne parametry

Sprawdzamy oba solvery na krańcach i poza krańcami zakresów parametrów (issue #12): czy wynik ma sens fizyczny,
czy solver jest stabilny i czy gęstość zostaje nieujemna ($u \ge 0$).

Zakresy z #10 (`units.model_ranges()`, jednostki modelu): $D$ od 0,11 do 32, $\rho$ od 0,23 do 15,8, $\beta$ na frakcję 2 Gy od 0,06 do 0,12.
Wychodzimy poza nie w obie strony: $D \to 0$, $\rho \to 0$, $\beta$ bardzo małe i bardzo duże, $r \to 0$, $r$ większe od guza
oraz przesunięcie $s$ większe od promienia guza (6).

Plan bazowy: pole pokrywa guz ($s = 0$, $r = 20$), $\beta = 1{,}5$, 10 frakcji co 0,4 od $t = 1$, obszar $L = 80$, $\Delta x = 0{,}1$.
Zmieniamy jeden parametr naraz.

In [ ]:
import time
import warnings
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.solvers import simulate, stable_dt
from offtarget.units import model_ranges

rng = seed_everything()

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
u0 = initial_condition(x)
base = Params(beta=1.5, s=0.0, r=20.0)
ranges = model_ranges()
print({k: tuple(round(v, 3) for v in r) for k, r in ranges.items()})

In [ ]:
cases = {
    "D": [0.0, 1e-3, ranges["D"][0], 1.0, ranges["D"][1]],
    "rho": [0.0, 1e-3, ranges["rho"][0], 1.0, ranges["rho"][1]],
    "beta": [1e-3, ranges["beta"][0], ranges["beta"][1], 1.5, 10.0, 50.0, 200.0],
    "r": [1e-3, 1.0, 3.0, 6.0, 20.0, 60.0],
    "s": [0.0, 3.0, 6.0, 9.0, 12.0, 20.0],
}
# dla przesunięcia bierzemy pole r = 8, jak w scenariuszu „pole przesunięte”
case_base = {"s": Params(beta=1.5, s=0.0, r=8.0)}


def check(p, method):
    start = time.perf_counter()
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore")
        u = simulate(p, schedule, x, t, u0=u0, method=method)
    elapsed = time.perf_counter() - start
    M = tumor_mass(u, x)
    finite = bool(np.all(np.isfinite(u)))
    m_min, _ = min_mass(t, M) if finite else (np.nan, np.nan)
    return u, {
        "min u": float(np.nanmin(u)) if finite else np.nan,
        "skończone": finite,
        "M_min / M(0)": m_min / M[0],
        "czas do odrostu": time_to_regrowth(t, M) if finite else np.nan,
        "M(10) / M(0)": M[-1] / M[0] if finite else np.nan,
        "czas obliczeń [s]": elapsed,
    }


runs, rows = {}, []
for name, values in cases.items():
    for v in values:
        p = replace(case_base.get(name, base), **{name: v})
        for method in ("rk4", "cn"):
            u, row = check(p, method)
            runs[(name, v, method)] = u
            rows.append({"parametr": name, "wartość": v, "solver": method, **row})
table = pd.DataFrame(rows)
table.to_csv(TABLES / "zad2_03_skrajne.csv", index=False)
table.style.format(precision=3)

Kolumna „min u” to najmniejsza gęstość w całym przebiegu (od $t = 0$, gdzie ogon plateau ma ok. $1{,}7 \cdot 10^{-15}$).
„Czas do odrostu” równy 0 oznacza, że masa nigdy nie spadła poniżej $M(0)$, a brak wartości, że guz nie odrósł do $t = 10$.

## Duże $\beta$ a krok RK4

W szczycie impulsu tempo zabijania wynosi $\beta / (w \sqrt{\pi}) \approx 11{,}3\,\beta$, a jawny RK4 jest stabilny dla
$\lambda\,\Delta t \le 2{,}78$. Dawny krok `stable_dt` brał pod uwagę tylko dyfuzję ($0{,}4\,\Delta x^2/D = 0{,}004$) i szerokość impulsu,
więc dla $D = 1$, $\Delta x = 0{,}1$ tracił stabilność przy $\beta \gtrsim 62$ (a dla $D = 0$ i kroku $w/5 = 0{,}01$ już przy $\beta \gtrsim 25$).
Teraz `stable_dt` ogranicza też $\lambda\,\Delta t \le 2$ dla zabijania i wzrostu. Poniżej porównanie z dawnym krokiem (`stable_dt(..., beta=0, rho=0)`).

In [ ]:
old_rows, old_runs = [], {}
for b in [10.0, 50.0, 70.0, 100.0, 200.0]:
    p = replace(base, beta=b)
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore")
        u_old = simulate(p, schedule, x, t, u0=u0, method="rk4", dt_max=stable_dt(dx, p.D, schedule))
    old_runs[b] = u_old
    M_old, M_cn = tumor_mass(u_old, x), tumor_mass(runs[("beta", b, "cn")], x) if ("beta", b, "cn") in runs else None
    if M_cn is None:
        M_cn = tumor_mass(simulate(p, schedule, x, t, u0=u0, method="cn"), x)
    finite = bool(np.all(np.isfinite(u_old)))
    old_rows.append({
        "beta": b,
        "krok nowy": stable_dt(dx, p.D, schedule, beta=p.beta, rho=p.rho),
        "dawny krok: skończone": finite,
        "dawny krok: min u": float(np.min(u_old)) if finite else np.nan,
        "dawny krok: M(10) / M(0)": M_old[-1] / M_old[0] if finite else np.nan,
        "solver B: M(10) / M(0)": M_cn[-1] / M_cn[0],
    })
old_table = pd.DataFrame(old_rows)
old_table.to_csv(TABLES / "zad2_03_krok_rk4.csv", index=False)
old_table.style.format(precision=4)

## Masa guza na krańcach zakresów (solver B)

In [ ]:
labels = {"D": "D", "rho": "ρ", "beta": "β", "r": "r (s = 0)", "s": "s (r = 8)"}
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8), sharey=True)
for ax, (name, values) in zip(axes, cases.items()):
    for v, c in zip(values, plt.cm.viridis(np.linspace(0, 0.9, len(values)))):
        M = tumor_mass(runs[(name, v, "cn")], x)
        ax.semilogy(t, np.maximum(M / M[0], 1e-8), color=c, lw=2, label=f"{labels[name].split()[0]} = {v:.3g}")
    ax.set(xlabel="czas t", title=labels[name], ylim=(1e-8, 20))
    ax.legend(fontsize=7)
axes[0].set_ylabel("M(t) / M(0)")
fig.tight_layout()
fig.savefig(FIGURES / "zad2_03_skrajne_masa.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
b = 70.0
u_old = old_runs[b]
k = int(np.argmin(u_old.min(axis=1)))
axes[0].plot(x, simulate(replace(base, beta=b), schedule, x, t, u0=u0, method="cn")[k], lw=2, label="solver B")
axes[0].plot(x, simulate(replace(base, beta=b), schedule, x, t, u0=u0, method="rk4")[k], "--", lw=2, label="RK4, nowy krok")
axes[0].plot(x, u_old[k], ":", lw=2, label="RK4, dawny krok")
axes[0].axhline(0, color="0.5", lw=0.8)
axes[0].set(xlabel="x", ylabel=f"u(x, t = {t[k]:g})", title=f"β = {b:g}, ujemne u przy dawnym kroku", xlim=(-15, 15))
axes[0].legend(fontsize=8)
for bb, c in zip([50.0, 70.0, 100.0], plt.cm.tab10.colors):
    M = tumor_mass(old_runs[bb], x)
    axes[1].semilogy(t, np.abs(M / M[0]), color=c, lw=2, label=f"β = {bb:g}, dawny krok")
axes[1].set(xlabel="czas t", ylabel="|M(t)| / M(0)", title="RK4 z dawnym krokiem (bez ograniczenia na β)")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_03_krok_rk4.png", dpi=150)

## Wnioski

- **W zakresach z literatury (#10) oba solvery są stabilne i zgodne.** Dla $D \in [0{,}11,\,32{,}2]$,
  $\rho \in [0{,}225,\,15{,}8]$ i $\beta \in [0{,}06,\,0{,}12]$ gęstość zostaje dodatnia, a `rk4` i `cn` dają te same wartości $M(t)$.
  Przy dużym $D = 32{,}2$ jawny RK4 zwalnia ~130 razy (6 s wobec 0,05 s dla solvera B), bo krok dyfuzji maleje jak $1/D$.
- **$D \to 0$ i $\rho \to 0$:** bez dyfuzji ($D = 0$) komórki nie uciekają z pola, a przy $D = 32{,}2$ szybka dyfuzja wypiera
  część guza poza $r = 20$ i powoduje odrost już w $t = 5{,}3$. Przy $\rho = 0$ masa po leczeniu zatrzymuje się na $4{,}5 \cdot 10^{-7}\,M(0)$;
  przy $\rho = 15{,}8$ wzrost między frakcjami przeważa nad $\beta = 1{,}5$ i masa w ogóle nie spada poniżej $M(0)$.
- **$\beta$ bardzo małe i bardzo duże:** dawki kliniczne ($\beta = 0{,}06\text{--}0{,}12$) w zaledwie 10 frakcjach nie zatrzymują wzrostu
  przy $\rho = 1$. Dla bardzo dużych $\beta$ ($\beta \ge 50$) stary krok RK4 (bez ograniczenia na tempo zabijania) daje ujemne gęstości
  (np. $\min u \approx -10^{-3}$ dla $\beta = 70$) i rozbiega się do `NaN` dla $\beta \ge 100$.
  Po dodaniu ograniczenia $\lambda\,\Delta t \le 2$ do `stable_dt` RK4 pozostaje stabilny (ujemne błędy zaokrągleń co najwyżej $10^{-20}$),
  a solver B (`cn`) dzięki dokładnemu krokowi reakcji zachowuje ścisłą dodatniość $u > 0$ nawet dla $\beta = 200$ przy stałym czasie ~0,05 s.
- **Geometria wiązki ($r$ i $s$):** dla $r \to 0$ leczenie nie działa ($M(10)/M(0) = 3{,}81$, jak bez leczenia);
  gdy $r$ przekracza promień guza (6), a jeszcze bardziej przy $r = 20$ i $60$, wiązka wycina także ogon dyfuzyjny.
  Przesunięcie $s$ większe od promienia guza (dla $r = 8$: $s = 9,\,12,\,20$) zostawia coraz większą część guza nietkniętą —
  przy $s = 20$ pole w ogóle nie trafia w guz w chwili startu, ale hamuje jego późniejszą ekspansję w prawo ($M(10)/M(0) = 3{,}39$ wobec $3{,}81$).